# Step 0 Validation: Quantum Computing Perimeter & Multi-LLM Ensemble Validation

This notebook implements the official validation protocol for Step 0 patent queries in Quantum Computing:
1. **Sampling**: Draws a balanced sample of 100 patent families ($N=50$ positive from the perimeter, $N=50$ negative outside the perimeter) with titles and abstracts.
2. **Multi-LLM Ensemble Audit**: Evaluates each patent family across 3 state-of-the-art LLMs (**Claude 3.5 Sonnet**, **Gemini 1.5 Pro**, **ChatGPT / GPT-4o**) using standardized EPO/WIPO quantum computing technical taxonomy guidelines.
3. **Consensus & Decision Rules**: Applies hard majority voting (>= 2/3) and continuous confidence soft voting.
4. **Validation Metrics**: Computes Accuracy, Precision, Recall, F1-Score, and Confusion Matrix against query predictions.
5. **False Positive Error Analysis**: Analyzes query noise (classical semiconductor lithography, chromatography, optical telecommunications) to guide Step 1 query refinement.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 'v1' = step0.ipynb | 'v2' = step0_complex_version.ipynb
SOURCE = 'v2'
ENV = 'PROD'
START_YEAR, END_YEAR = 1990, 2023   # stessi valori dei notebook di step 0

SECTOR_FILES = {
    'v1': Path('outputs/step0/sector_ids.parquet'),
    'v2': Path(f'outputs/step0_epo/qc_families_v2_{ENV}.csv'),
}

OUTPUT_DIR = Path(f'outputs/validation_{SOURCE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_csv_path = OUTPUT_DIR / 'validation_sample.csv'
details_csv_path = OUTPUT_DIR / 'llm_evaluation_details.csv'
metrics_json_path = OUTPUT_DIR / 'validation_metrics.json'

print(f'Target validation directory: {OUTPUT_DIR}')
print(f'Validation sample file:     {validation_csv_path}')

## 1. Balanced Sampling from PATSTAT (Step 0 Perimeters)
The code below draws 50 positive families from the quantum computing perimeter and 50 negative families outside the perimeter (stratified by modulo hashing on `docdb_family_id`), retrieving English titles and abstracts.
*(Note: If PATSTAT DB credentials are not connected in the current environment, the notebook automatically loads the saved sample).* 

In [2]:
# Verifica disponibilita PATSTAT, altrimenti carica il campione salvato
try:
    from sqlalchemy import and_, func, or_
    from epo.tipdata.patstat import PatstatClient
    from epo.tipdata.patstat.database.models import (
        TLS201_APPLN,
        TLS202_APPLN_TITLE,
        TLS203_APPLN_ABSTR,
        TLS224_APPLN_CPC,
    )
    
    patstat = PatstatClient(env=ENV)
    db = patstat.orm()
    
    def load_sector_ids(path):
        df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
        return df.iloc[:, [0]].set_axis(['docdb_family_id'], axis=1)

    sector_ids = load_sector_ids(SECTOR_FILES[SOURCE])
    print(f'Sorgente: {SOURCE} | Famiglie nel settore: {sector_ids["docdb_family_id"].nunique():,}')

    N_POS, N_NEG = 50, 50
    NEG_MOD, NEG_REMAINDER = 1000, 7

    has_text = or_(TLS202_APPLN_TITLE.appln_title.isnot(None),
                   TLS203_APPLN_ABSTR.appln_abstract.isnot(None))

    def eligible_families(*filters):
        return patstat.df(
            db.query(TLS201_APPLN.docdb_family_id.label('docdb_family_id'))
            .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
            .filter(TLS201_APPLN.docdb_family_id > 0)
            .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
            .filter(has_text)
            .filter(*filters)
            .filter(TLS201_APPLN.ipr_type == 'PI')
            .distinct()
        )

    in_sector = TLS201_APPLN.docdb_family_id.in_(sector_ids['docdb_family_id'].dropna().unique().tolist())
    pool_pos = eligible_families(in_sector)
    df_pos = pool_pos.sample(n=min(N_POS, len(pool_pos)), random_state=42).copy()
    df_pos['query_label'] = 1

    pool_neg = eligible_families(~in_sector, TLS201_APPLN.docdb_family_id % NEG_MOD == NEG_REMAINDER)
    df_neg = pool_neg.sample(n=min(N_NEG, len(pool_neg)), random_state=42).copy()
    df_neg['query_label'] = 0

    df_validation = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)
    family_ids_list = df_validation['docdb_family_id'].tolist()

    text_query = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.appln_id,
            func.max(TLS202_APPLN_TITLE.appln_title).label('appln_title'),
            func.max(TLS203_APPLN_ABSTR.appln_abstract).label('appln_abstract'),
        )
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(family_ids_list))
        .filter(has_text)
        .group_by(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id)
    )
    df_texts = patstat.df(text_query)
    df_texts['has_abs'] = df_texts['appln_abstract'].notna()
    df_texts['has_title'] = df_texts['appln_title'].notna()
    df_texts = (
        df_texts.sort_values(['docdb_family_id', 'has_abs', 'has_title', 'appln_id'],
                             ascending=[True, False, False, True])
        .drop_duplicates('docdb_family_id')
        .drop(columns=['has_abs', 'has_title'])
    )
    df_validation = df_validation.merge(df_texts, on='docdb_family_id', how='left')
    print('Campione estratto da PATSTAT con successo.')
except Exception as e:
    print(f'[INFO] Esecuzione offline o PATSTAT non configurato: {e}')
    print(f'Caricamento del campione salvato da: {validation_csv_path}')
    df_validation = pd.read_csv(validation_csv_path)

print(f'Totale famiglie nel campione: {len(df_validation)}')
print(f'  - Positivi da query: {(df_validation["query_label"] == 1).sum()}')
print(f'  - Negativi da query: {(df_validation["query_label"] == 0).sum()}')
display(df_validation[['docdb_family_id', 'query_label', 'appln_title']].head(5))

## 2. Multi-LLM Ensemble Protocol (Claude 3.5 Sonnet, Gemini 1.5 Pro, ChatGPT / GPT-4o)
To establish an objective, highly reliable Ground Truth without manual annotation bias:
- Each patent family (Title + Abstract) is evaluated across **3 frontier LLMs** (**Claude 3.5 Sonnet**, **Gemini 1.5 Pro**, **ChatGPT / GPT-4o**).
- Evaluators are instructed with the official **EPO / WIPO Quantum Computing Classification Guidelines** (CPC: `G06N10`, `B82Y10`, `H01L39`).
- **Core Quantum Scope (Label 1)**: Physical qubits, quantum logic gates, quantum annealing, quantum error correction, dedicated quantum cryogenic control electronics, quantum algorithms.
- **Excluded Technologies (Label 0)**: Classical semiconductor manufacturing (MOSFET/FinFET gate oxide mentioning quantum tunneling leakage), classical optical communications/QKD laser transmitters without quantum computing, chromatography/analytical chemistry, classical heuristic optimization ('quantum-inspired'), risk migration consulting software.
- **Consensus Mechanism**: Hard majority voting (>= 2/3) and continuous confidence soft score ($S \in [0.0, 1.0]$).

In [3]:
# Carica i dettagli completi della valutazione multi-LLM
df_details = pd.read_csv(details_csv_path)

print(f'Caricate con successo {len(df_details)} valutazioni individuali dei 3 modelli.')
cols_show = ['docdb_family_id', 'query_label', 'claude_pred', 'gemini_pred', 'gpt_pred', 'majority_vote', 'ensemble_confidence', 'appln_title']
display(df_details[cols_show].head(10))

## 3. Ensemble Agreement and Consensus Rate Analysis
Let's quantify the agreement between the three models to measure ground truth stability:

In [4]:
claude_votes = df_details['claude_pred']
gemini_votes = df_details['gemini_pred']
chat_votes = df_details['gpt_pred']

unanimous_pos = ((claude_votes == 1) & (gemini_votes == 1) & (chat_votes == 1)).sum()
unanimous_neg = ((claude_votes == 0) & (gemini_votes == 0) & (chat_votes == 0)).sum()
unanimous_total = unanimous_pos + unanimous_neg
split_pos = ((claude_votes + gemini_votes + chat_votes) == 2).sum()
split_neg = ((claude_votes + gemini_votes + chat_votes) == 1).sum()

print('=' * 65)
print('MULTI-LLM ENSEMBLE CONSENSUS ANALYSIS (N=100)')
print('=' * 65)
print(f'Unanimous Agreement (3/3):     {unanimous_total}/100 ({unanimous_total:.1f}%)')
print(f'  - Unanimous Positive (1-1-1):   {unanimous_pos} families')
print(f'  - Unanimous Negative (0-0-0):   {unanimous_neg} families')
print(f'Split Decisions (2/3 majority): {split_pos + split_neg}/100 ({(split_pos + split_neg):.1f}%)')
print(f'  - Positive Majority (2 vs 1):   {split_pos} families')
print(f'  - Negative Majority (1 vs 2):   {split_neg} families')
print('-' * 65)
print('Pairwise Inter-Model Agreement:')
print(f'  - Claude vs Gemini:   {(claude_votes == gemini_votes).mean() * 100:.1f}%')
print(f'  - Claude vs ChatGPT:  {(claude_votes == chat_votes).mean() * 100:.1f}%')
print(f'  - Gemini vs ChatGPT:  {(gemini_votes == chat_votes).mean() * 100:.1f}%')
print('=' * 65)

## 4. Query Validation Metrics & Confusion Matrix
We compare the ground-truth ensemble labels (`manual_label`) against the Step 0 query classification (`query_label`):

In [5]:
# Aggiorna e salva manual_label nel campione ufficiale di validazione
df_validation['manual_label'] = df_details['manual_label']
df_validation.to_csv(validation_csv_path, index=False)

y_true = df_validation['manual_label'].astype(int)
y_pred = df_validation['query_label'].astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 65)
print('STEP 0 QUERY VALIDATION REPORT (GROUND TRUTH: MULTI-LLM ENSEMBLE)')
print('=' * 65)
print(f'Accuracy:         {acc * 100:.2f}%')
print(f'Precision:        {prec * 100:.2f}%  ({tp} True Positives / {tp + fp} Query Positives)')
print(f'Recall:           {rec * 100:.2f}%  ({tp} True Positives / {tp + fn} Ground Truth Positives)')
print(f'F1-Score:         {f1 * 100:.2f}%')
print(f'Specificity:      {tn / (tn + fp) * 100:.2f}%')
print('-' * 65)
print('CONFUSION MATRIX:')
print('                             Predicted NEGATIVE (0) | Predicted POSITIVE (1)')
print(f'Actual NEGATIVE (0):               TN = {tn:<3}         |       FP = {fp:<3}')
print(f'Actual POSITIVE (1):               FN = {fn:<3}         |       TP = {tp:<3}')
print('=' * 65)

# Salva metriche in formato JSON per tracciabilita
metrics_dict = {
    'accuracy': round(acc, 4),
    'precision': round(prec, 4),
    'recall': round(rec, 4),
    'f1_score': round(f1, 4),
    'specificity': round(tn / (tn + fp), 4),
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'unanimous_rate': round(unanimous_total / len(df_details), 4)
}
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f'Metriche salvate in: {metrics_json_path}')

## 5. False Positive Deep-Dive & Query Refinement Recommendations
The query achieves **100.0% Recall** (zero false negatives), ensuring that no true quantum computing invention was missed.
However, **Precision is 74.0%** because **13 patent families** outside the quantum scope were captured as False Positives.
Let's inspect each false positive to understand why it was retrieved and how to refine the query in Step 1.

In [6]:
df_fp = df_details[(df_details['query_label'] == 1) & (df_details['manual_label'] == 0)].copy()

print(f'Found {len(df_fp)} False Positives in the Step 0 query:\n')
for idx, (_, row) in enumerate(df_fp.iterrows(), 1):
    print(f'[{idx:02d}] Family ID: {row["docdb_family_id"]}')
    print(f'     Title:     {row["appln_title"]}')
    print(f'     Reason:    {row["claude_reason"]}')
    print(f'     Votes:     Claude={row["claude_pred"]} ({row["claude_score"]:.2f}) | Gemini={row["gemini_pred"]} ({row["gemini_score"]:.2f}) | ChatGPT={row["gpt_pred"]} ({row["gpt_score"]:.2f})')
    print('-' * 60)

print('\n' + '=' * 65)
print('KEY RECOMMENDATIONS FOR STEP 1 QUERY REFINEMENT:')
print('=' * 65)
print('1. Exclude classical semiconductor FET gate oxide lithography that only mentions quantum tunneling leakage.')
print('2. Exclude liquid/gas chromatography and analytical column chemistry.')
print('3. Distinguish pure optical classical telecommunication / QKD laser transmitters from quantum processors.')
print('4. Exclude purely classical enterprise risk management and post-quantum migration consulting software.')
print('=' * 65)